In [84]:
import pandas as pd
import matplotlib.pyplot as plt

In [85]:
def style_table(df, caption=""):
    """Apply consistent blue-header styling to a DataFrame for display."""
    styled = df.style\
        .set_properties(**{
            "font-size": "12px",
            "font-weight": "bold",
            "border": "1px solid #ddd",
            "padding": "6px 12px",
            "text-align": "center"
        })\
        .set_table_styles([
            {"selector": "th", "props": [
                ("background-color", "#2196F3"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("padding", "6px 12px"),
                ("text-align", "center")
            ]},
            {"selector": "tr:nth-child(even)", "props": [
                ("background-color", "#f2f2f2")
            ]},
        ])\
        .hide(axis="index")
    if caption:
        styled = styled.set_caption(caption)
    return styled

def export_table_png(df, filepath, title=""):
    # Calculate column widths based on longest content in each column
    col_widths = []
    for col in df.columns:
        max_content = max(
            len(str(col)),
            df[col].astype(str).map(len).max()
        )
        col_widths.append(max_content)
    
    # Total figure width proportional to content
    fig_width = max(sum(col_widths) * 0.15, 6)
    fig_height = len(df) * 0.6 + 0.8

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.6)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor("#2196F3")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#f2f2f2")
        else:
            cell.set_facecolor("white")
        cell.set_edgecolor("#dddddd")
    if title:
        ax.set_title(title, fontsize=12, y=0.88)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved to {filepath}")

In [86]:
data_overview = {
    "Sample":    ["KM1, KM2, KM3", "KM4, KM5, KM6", "KM7, KM8, KM9"],
    "Condition": ["CT8", "CT20", "PerKO"],
    "Biological meaning": [
        "WT Day",
        "WT Night",
        "clock disrupted",
    ],
}

df_overview = pd.DataFrame(data_overview)

display(style_table(df_overview, "Table 1 — Sample overview"))

export_table_png(
    df_overview,
    "../../figures/plots/table1_sample_overview.png",
    title="Table 1 — Sample overview"
)

Sample,Condition,Biological meaning
"KM1, KM2, KM3",CT8,WT Day
"KM4, KM5, KM6",CT20,WT Night
"KM7, KM8, KM9",PerKO,clock disrupted


Saved to ../../figures/plots/table1_sample_overview.png


In [87]:
SALMON_DIR = "/Users/gricey/Desktop/Internship/data/salmon" # location of salmon files
samples = [f"KM{i}" for i in range(1, 10)] # list of sample names: KM1, KM2, ..., KM9

rows = [] # create an empty list to store the summary data

for sample in samples:
    path = f"{SALMON_DIR}/{sample}_quant/quant.sf" # construct file path
    df = pd.read_csv(path, sep="\t") # read the quant.sf file into a DataFrame
    total_transcripts = len(df) # counts the number of rows in the DataFrame
    total_reads = df["NumReads"].sum() # sums "NumReads" column to get total mapped reads
    
    rows.append({
        "Sample": sample,
        "Total transcripts quantified": total_transcripts,
        "Total mapped reads": int(total_reads),
    })

df_summary = pd.DataFrame(rows)
display(style_table(df_summary, "Table 2 — Salmon quantification summary"))

export_table_png(df_summary,
                 "../../figures/plots/table2_quantification_summary.png",
                 "Table 2 — Salmon quantification summary")

print(f"\nMean reads per sample: {df_summary['Total mapped reads'].mean():,.0f}")
print(f"Min: {df_summary['Total mapped reads'].min():,.0f}")
print(f"Max: {df_summary['Total mapped reads'].max():,.0f}")

Sample,Total transcripts quantified,Total mapped reads
KM1,155263,9704275
KM2,155263,16068247
KM3,155263,13311617
KM4,155263,17859661
KM5,155263,18611438
KM6,155263,12115765
KM7,155263,6575980
KM8,155263,14812353
KM9,155263,17464840


Saved to ../../figures/plots/table2_quantification_summary.png

Mean reads per sample: 14,058,242
Min: 6,575,980
Max: 18,611,438
